In [12]:
from dotenv import load_dotenv
import os
from openai import OpenAI
import json
from concurrent.futures import ThreadPoolExecutor, as_completed

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
print("Client ready.")

Client ready.


In [13]:
def call_openai(system_prompt, user_message, model="gpt-4o-mini"):
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_message}
        ]
    )
    return response.choices[0].message.content

## Step 2: Initial Prompts (Version 1 — Zero-Shot)

Three basic prompts, one test each. No optimization yet — we want to see raw output before any engineering.

In [14]:
# Task 1: Sentiment Analysis — Version 1
# Zero-shot, no format constraints, no output schema

sentiment_prompt_v1 = """
Classify this customer message: "I love this product! It's exactly what I needed."
"""

result = call_openai("You are a helpful assistant.", sentiment_prompt_v1)
print("Task 1 — Sentiment Analysis v1:")
print(result)

Task 1 — Sentiment Analysis v1:
The customer message can be classified as "Positive Feedback" or "Customer Satisfaction."


In [15]:
# Task 3: Data Extraction — Version 1
# Zero-shot, no schema, no format requirement

extraction_prompt_v1 = """
Extract information from this customer feedback: "I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged."
"""

result = call_openai("You are a helpful assistant.", extraction_prompt_v1)
print("Task 3 — Data Extraction v1:")
print(result)

Task 3 — Data Extraction v1:
Here is the extracted information from the customer feedback:

- Item number: #12345
- Order date: March 15th
- Delivery: Fast
- Packaging condition: Damaged


In [20]:
product_prompt_v1 = """
Create a product description for a wireless mouse that costs $29.99.
"""

## Step 3: Systematic Testing — 5 Runs

Run each v1 prompt 5 times to observe consistency. Are responses the same format? Same content? What varies?

In [19]:
def run_prompt_n_times(system_prompt, user_prompt, n=5):
    """Run the same prompt n times and return all results."""
    results = []
    for i in range(n):
        result = call_openai(system_prompt, user_prompt)
        results.append(result)
        print(f"Run {i+1}: {result}")
    
    unique = len(set(results))
    print(f"\n📊 Unique outputs: {unique}/{n}")
    print(f"📊 Consistency rate: {((n - unique + 1) / n * 100):.0f}%")
    return results

print("✅ run_prompt_n_times loaded")

✅ run_prompt_n_times loaded


In [17]:
print("="*60)
print("Task 1 — Sentiment Analysis v1 — 5 runs")
print("="*60)
sentiment_5_runs = run_prompt_n_times(
    "You are a helpful assistant.",
    sentiment_prompt_v1,
    n=5
)

Task 1 — Sentiment Analysis v1 — 5 runs
Run 1: The customer message can be classified as "Positive feedback" or "Customer satisfaction."
Run 2: This customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.
Run 3: The customer message can be classified as **positive feedback** or **satisfaction**.
Run 4: The customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.
Run 5: The customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.

📊 Unique outputs: 4/5
📊 Consistency rate: 40%


In [21]:
print("="*60)
print("Task 2 — Product Description v1 — 5 runs")
print("="*60)
product_5_runs = run_prompt_n_times(
    "You are a helpful assistant.",
    product_prompt_v1,
    n=5
)

Task 2 — Product Description v1 — 5 runs
Run 1: **Product Description: Wireless Comfort Mouse - $29.99**

Say goodbye to tangled cords and hello to seamless navigation with our Wireless Comfort Mouse. Designed for productivity and comfort, this sleek and stylish mouse is the perfect companion for your laptop or desktop, whether at home, in the office, or on the go.

**Key Features:**

- **Ergonomic Design:** The Wireless Comfort Mouse features a contoured shape that fits snugly in your hand, reducing strain and enhancing comfort during extended use.

- **Reliable Wireless Connection:** Enjoy a clutter-free workspace with advanced wireless technology that provides a stable connection up to 33 feet away. No more interruptions — just smooth, accurate tracking.

- **High Precision Tracking:** Equipped with a high-resolution optical sensor, this mouse delivers precise cursor control on a variety of surfaces, making it ideal for both everyday tasks and detailed projects.

- **Long Battery Li

In [ ]:
print("="*60)
print("Task 3 — Data Extraction v1 — 5 runs")
print("="*60)
extraction_5_runs = run_prompt_n_times(
    "You are a helpful assistant.",
    extraction_prompt_v1,
    n=5
)

Task 3 — Data Extraction v1 — 5 runs
Run 1: Here is the extracted information from the customer feedback:

- **Order Number**: 12345
- **Order Date**: March 15th
- **Delivery Speed**: Fast
- **Packaging Condition**: Damaged
Run 2: Here are the extracted details from the customer feedback:

- Order Number: #12345
- Order Date: March 15th
- Delivery Experience: Fast
- Packaging Condition: Damaged
Run 3: Here is the extracted information from the customer feedback:

- **Order Number:** 12345
- **Order Date:** March 15th
- **Delivery:** Fast
- **Packaging Condition:** Damaged
Run 4: Here’s the extracted information from the customer feedback:

- Order Number: #12345
- Order Date: March 15th
- Delivery: Fast
- Packaging Condition: Damaged
Run 5: Here is the extracted information from the customer feedback:

- **Order Number:** #12345
- **Order Date:** March 15th
- **Delivery Speed:** Fast
- **Packaging Condition:** Damaged

📊 Unique outputs: 5/5
📊 Consistency rate: 20%


## Step 4: Systematic Testing — 10 Runs

Increasing to 10 runs to see if new failure patterns emerge.

In [22]:
print("="*60)
print("Task 1 — Sentiment Analysis v1 — 10 runs")
print("="*60)
sentiment_10_runs = run_prompt_n_times(
    "You are a helpful assistant.",
    sentiment_prompt_v1,
    n=10
)

Task 1 — Sentiment Analysis v1 — 10 runs
Run 1: The customer message can be classified as "Positive Feedback" or "Customer Satisfaction."
Run 2: The customer message can be classified as **Positive Feedback** or **Customer Satisfaction**.
Run 3: The customer message can be classified as "Positive Feedback" or "Customer Satisfaction."
Run 4: The customer message can be classified as "Positive Feedback" or "Satisfaction."
Run 5: This customer message can be classified as "Positive Feedback" or "Satisfaction."
Run 6: The customer message can be classified as **Positive Feedback** or **Customer Praise**.
Run 7: This customer message is positive feedback.
Run 8: The customer message can be classified as "Positive Feedback" or "Customer Satisfaction."
Run 9: The customer message can be classified as "Positive Feedback" or "Satisfaction" since the customer expresses their love for the product and confirms it meets their needs.
Run 10: This customer message can be classified as **positive feed

In [23]:
print("="*60)
print("Task 2 — Product Description v1 — 10 runs")
print("="*60)
product_10_runs = run_prompt_n_times(
    "You are a helpful assistant.",
    product_prompt_v1,
    n=10
)

Task 2 — Product Description v1 — 10 runs
Run 1: **Product Description: Wireless Precision Mouse**

Elevate your everyday computing experience with our Wireless Precision Mouse, designed to deliver exceptional performance and comfort at an unbeatable price of just $29.99. Whether you're working from home, gaming, or browsing the web, this versatile mouse is the perfect companion for all your digital tasks.

**Key Features:**

- **Seamless Wireless Connectivity:** Enjoy the freedom of a clutter-free workspace with advanced 2.4GHz wireless technology. Simply plug in the USB receiver, and you're ready to go—no tangled wires, no hassle!

- **Ergonomic Design:** Crafted for comfort, the Wireless Precision Mouse features an ergonomic shape that fits snugly in your hand, reducing strain during extended use. The textured grip ensures optimal control, making it ideal for both office work and casual gaming.

- **High Precision Tracking:** With a DPI range adjustable from 800 to 2400, this mouse 

In [24]:
print("="*60)
print("Task 3 — Data Extraction v1 — 10 runs")
print("="*60)
extraction_10_runs = run_prompt_n_times(
    "You are a helpful assistant.",
    extraction_prompt_v1,
    n=10
)

Task 3 — Data Extraction v1 — 10 runs
Run 1: Here is the extracted information from the customer feedback:

- **Order Number:** #12345
- **Order Date:** March 15th
- **Delivery:** Fast
- **Packaging Condition:** Damaged
Run 2: Here is the extracted information from the customer feedback:

- **Order Number**: 12345
- **Order Date**: March 15th
- **Delivery Speed**: Fast
- **Packaging Condition**: Damaged
Run 3: Here's the extracted information from the customer feedback:

- **Order Number**: #12345
- **Order Date**: March 15th
- **Delivery Speed**: Fast
- **Packaging Condition**: Damaged
Run 4: Customer Feedback Summary:
- Order Number: #12345
- Order Date: March 15th
- Delivery Speed: Fast
- Issue: Damaged packaging
Run 5: Here's the extracted information from the customer feedback:

- **Item Ordered**: #12345
- **Order Date**: March 15th
- **Delivery Speed**: Fast
- **Issue**: Damaged packaging
Run 6: Here’s the extracted information from the customer feedback:

- **Order Number**: #1

## Step 5: Systematic Testing — 15 Runs + Failure Analysis

Final baseline test. After this we have enough data to document failure patterns before iterating.

In [25]:
print("="*60)
print("Task 1 — Sentiment Analysis v1 — 15 runs")
print("="*60)
sentiment_15_runs = run_prompt_n_times(
    "You are a helpful assistant.",
    sentiment_prompt_v1,
    n=15
)

Task 1 — Sentiment Analysis v1 — 15 runs
Run 1: This customer message can be classified as positive feedback or a satisfied customer review.
Run 2: The customer message can be classified as **positive feedback**.
Run 3: This customer message can be classified as **Positive Feedback** or **Customer Appreciation**.
Run 4: The customer message can be classified as **positive feedback** or **customer satisfaction**.
Run 5: The customer message can be classified as **positive feedback** or **customer satisfaction**.
Run 6: The customer message can be classified as "Positive Feedback" or "Customer Satisfaction."
Run 7: The customer message can be classified as **positive feedback** or **satisfaction**.
Run 8: This customer message can be classified as "Positive Feedback" or "Customer Satisfaction."
Run 9: The customer message can be classified as positive feedback or a customer satisfaction message.
Run 10: The customer message can be classified as "Positive Feedback" or "Customer Satisfacti

In [26]:
print("="*60)
print("Task 2 — Product Description v1 — 15 runs")
print("="*60)
product_15_runs = run_prompt_n_times(
    "You are a helpful assistant.",
    product_prompt_v1,
    n=15
)

Task 2 — Product Description v1 — 15 runs
Run 1: **Product Description: Wireless Freedom Mouse**

Elevate your productivity with the Wireless Freedom Mouse, expertly designed for seamless functionality and comfort. Priced at just $29.99, this sleek, modern mouse is the perfect accessory for home, office, or on-the-go use.

**Key Features:**

- **Wireless Connectivity:** Enjoy the freedom of movement without the hassle of tangled cords. The advanced 2.4GHz wireless technology ensures a reliable connection with a range of up to 33 feet, allowing you to work from anywhere in your workspace.

- **Ergonomic Design:** Designed with your comfort in mind, the Wireless Freedom Mouse features a contoured shape that fits naturally in your hand, reducing strain even during long hours of use. Its textured grip ensures a secure hold, providing maximum control and comfort.

- **Smooth Navigation:** Equipped with an optical sensor, this mouse delivers precise tracking on various surfaces, so you can n

In [27]:
print("="*60)
print("Task 3 — Data Extraction v1 — 15 runs")
print("="*60)
extraction_15_runs = run_prompt_n_times(
    "You are a helpful assistant.",
    extraction_prompt_v1,
    n=15
)

Task 3 — Data Extraction v1 — 15 runs
Run 1: Here is the extracted information from the customer feedback:

- **Item Ordered**: #12345
- **Order Date**: March 15th
- **Delivery Speed**: Fast
- **Packaging Condition**: Damaged
Run 2: Here is the extracted information from the customer feedback:

- **Order Number**: #12345
- **Order Date**: March 15th
- **Delivery Experience**: Fast
- **Packaging Condition**: Damaged
Run 3: Here is the extracted information from the customer feedback:

- **Order number**: #12345
- **Order date**: March 15th
- **Delivery**: Fast
- **Packaging condition**: Damaged
Run 4: Here is the extracted information from the customer feedback:

- **Order item number**: #12345
- **Order date**: March 15th
- **Delivery review**: Fast
- **Packaging issue**: Damaged
Run 5: Here is the extracted information from the customer feedback:

- **Item Number:** #12345
- **Order Date:** March 15th
- **Delivery Feedback:** Fast
- **Packaging Condition:** Damaged
Run 6: Customer Fee

In [28]:
print("""
╔══════════════════════════════════════════════════════════════════════╗
║                    FAILURE ANALYSIS — v1 PROMPTS                    ║
╠══════════════╦═══════════╦══════════════════════════════════════════╣
║ Task         ║ Consist.  ║ Failure Patterns                         ║
╠══════════════╬═══════════╬══════════════════════════════════════════╣
║ Sentiment    ║ ~30%      ║ - No fixed output format                 ║
║              ║           ║ - Mixes labels: "Positive Feedback",     ║
║              ║           ║   "Satisfaction", "positive feedback"    ║
║              ║           ║ - Returns sentences instead of labels    ║
║              ║           ║ - Inconsistent use of quotes/bold        ║
╠══════════════╬═══════════╬══════════════════════════════════════════╣
║ Product      ║ ~10%      ║ - Variable length (100 to 400+ words)    ║
║ Description  ║           ║ - Invents specs not in the prompt        ║
║              ║           ║ - Inconsistent structure and headers     ║
║              ║           ║ - Tone varies: formal, casual, salesy    ║
╠══════════════╬═══════════╬══════════════════════════════════════════╣
║ Data         ║ ~20%      ║ - Field names change every run           ║
║ Extraction   ║           ║ - No consistent schema                   ║
║              ║           ║ - Mix of bullet points and prose         ║
║              ║           ║ - Unparseable by downstream systems      ║
╚══════════════╩═══════════╩══════════════════════════════════════════╝

ROOT CAUSE: All three prompts lack output format constraints, role
definition, and schema. The model has no guardrails — it improvises
every response.
""")


╔══════════════════════════════════════════════════════════════════════╗
║                    FAILURE ANALYSIS — v1 PROMPTS                    ║
╠══════════════╦═══════════╦══════════════════════════════════════════╣
║ Task         ║ Consist.  ║ Failure Patterns                         ║
╠══════════════╬═══════════╬══════════════════════════════════════════╣
║ Sentiment    ║ ~30%      ║ - No fixed output format                 ║
║              ║           ║ - Mixes labels: "Positive Feedback",     ║
║              ║           ║   "Satisfaction", "positive feedback"    ║
║              ║           ║ - Returns sentences instead of labels    ║
║              ║           ║ - Inconsistent use of quotes/bold        ║
╠══════════════╬═══════════╬══════════════════════════════════════════╣
║ Product      ║ ~10%      ║ - Variable length (100 to 400+ words)    ║
║ Description  ║           ║ - Invents specs not in the prompt        ║
║              ║           ║ - Inconsistent structure and head

## Part 3: Iteration 1 — Improved Prompts (Version 2)

Based on failure analysis, v2 adds:
- Explicit output format requirements
- Clear constraints
- Role definition

No few-shot examples yet — that comes in v3.

In [29]:
# Task 1: Sentiment Analysis — Version 2
# Improvements: role, single-word constraint, explicit label options

sentiment_prompt_v2 = """
You are a customer service analyst.

Classify the sentiment of the customer message below.
Respond with ONLY one word: POSITIVE, NEGATIVE, or NEUTRAL.
No explanation. No punctuation. Just the label.

Customer message: "I love this product! It's exactly what I needed."
"""

result = call_openai("You are a customer service analyst.", sentiment_prompt_v2)
print("Task 1 — Sentiment Analysis v2:")
print(result)

Task 1 — Sentiment Analysis v2:
POSITIVE


In [30]:
print("="*60)
print("Task 1 — Sentiment Analysis v2 — 15 runs")
print("="*60)
sentiment_v2_runs = run_prompt_n_times(
    "You are a customer service analyst.",
    sentiment_prompt_v2,
    n=15
)

Task 1 — Sentiment Analysis v2 — 15 runs
Run 1: POSITIVE
Run 2: POSITIVE
Run 3: POSITIVE
Run 4: POSITIVE
Run 5: POSITIVE
Run 6: POSITIVE
Run 7: POSITIVE
Run 8: POSITIVE
Run 9: POSITIVE
Run 10: POSITIVE
Run 11: POSITIVE
Run 12: POSITIVE
Run 13: POSITIVE
Run 14: POSITIVE
Run 15: POSITIVE

📊 Unique outputs: 1/15
📊 Consistency rate: 100%


In [31]:
# Task 2: Product Description — Version 2
# Improvements: structure requirement, length constraint, tone guidance

product_prompt_v2 = """
You are a product copywriter.

Write a product description for a wireless mouse that costs $29.99.

Requirements:
- Length: 60-80 words maximum
- Structure: one paragraph, no bullet points
- Tone: professional and concise
- Do not invent specifications not provided
"""

result = call_openai("You are a product copywriter.", product_prompt_v2)
print("Task 2 — Product Description v2:")
print(result)

Task 2 — Product Description v2:
Experience seamless navigation with our premium wireless mouse, designed for both comfort and efficiency. Priced at just $29.99, this sleek accessory features precise tracking for effortless scrolling and pinpoint accuracy. Its ergonomic design ensures a comfortable grip during extended use, while the wireless functionality keeps your workspace clutter-free. Enjoy reliable connectivity and long battery life, making this mouse the perfect companion for home, office, or on-the-go productivity. Elevate your computing experience today!


In [32]:
print("="*60)
print("Task 2 — Product Description v2 — 15 runs")
print("="*60)
product_v2_runs = run_prompt_n_times(
    "You are a product copywriter.",
    product_prompt_v2,
    n=15
)

Task 2 — Product Description v2 — 15 runs
Run 1: Experience seamless productivity with our wireless mouse, designed for comfort and efficiency at just $29.99. Its ergonomic shape fits snugly in your hand, reducing fatigue during long work sessions. Enjoy the freedom of wireless connectivity, allowing you to navigate effortlessly from anywhere in the room. The sleek design and responsive performance make it an ideal companion for both office and home settings. Elevate your work experience today with this essential tool.
Run 2: Introducing our sleek and efficient wireless mouse, designed for seamless navigation and productivity. Priced at just $29.99, it features an ergonomic shape that fits comfortably in your hand, ensuring hours of use without fatigue. With a reliable wireless connection, enjoy freedom of movement without the clutter of cords. Its compact design makes it perfect for on-the-go professionals and gamers alike. Upgrade your workspace with this essential tool for optimal p

In [35]:
# Task 3: Data Extraction — Version 2
# Improvements: fixed field names, structured output, null handling

extraction_prompt_v2 = """
You are a data extraction specialist.

Extract information from the customer feedback below.
Return ONLY a JSON object with these exact fields:
{
  "order_number": "string or null",
  "order_date": "string or null",
  "delivery_speed": "fast | slow | normal | null",
  "packaging_condition": "damaged | good | null"
}

No explanation. No markdown. Just the JSON.

Customer feedback: "I ordered item #12345 on March 15th. The delivery was fast but the packaging was damaged."
"""

result = call_openai("You are a data extraction specialist.", extraction_prompt_v2)
print("Task 3 — Data Extraction v2:")
print(result)

Task 3 — Data Extraction v2:
{
  "order_number": "12345",
  "order_date": "March 15th",
  "delivery_speed": "fast",
  "packaging_condition": "damaged"
}


In [34]:
print("="*60)
print("Task 3 — Data Extraction v2 — 15 runs")
print("="*60)
extraction_v2_runs = run_prompt_n_times(
    "You are a data extraction specialist.",
    extraction_prompt_v2,
    n=15
)

Task 3 — Data Extraction v2 — 15 runs
Run 1: {
  "order_number": "12345",
  "order_date": "March 15th",
  "delivery_speed": "fast",
  "packaging_condition": "damaged"
}
Run 2: {
  "order_number": "12345",
  "order_date": "March 15th",
  "delivery_speed": "fast",
  "packaging_condition": "damaged"
}
Run 3: {
  "order_number": "12345",
  "order_date": "March 15th",
  "delivery_speed": "fast",
  "packaging_condition": "damaged"
}
Run 4: {
  "order_number": "12345",
  "order_date": "March 15th",
  "delivery_speed": "fast",
  "packaging_condition": "damaged"
}
Run 5: {
  "order_number": "12345",
  "order_date": "March 15th",
  "delivery_speed": "fast",
  "packaging_condition": "damaged"
}
Run 6: {
  "order_number": "12345",
  "order_date": "March 15th",
  "delivery_speed": "fast",
  "packaging_condition": "damaged"
}
Run 7: {
  "order_number": "12345",
  "order_date": "March 15th",
  "delivery_speed": "fast",
  "packaging_condition": "damaged"
}
Run 8: {
  "order_number": "12345",
  "order_